<!-- parity-note -->
## MATLAB Parity Note
- Source MATLAB helpfile: `DecodingExampleWithHist.mlx`
- Fidelity status: `exact`
- Remaining justified differences: Mirrors the MATLAB history-aware decoding workflow. Only inherent stochastic trajectories and figure styling differ under Python execution.

In [ ]:
# nSTAT-python notebook example: DecodingExampleWithHist
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
SRC_PATH = (REPO_ROOT / "src").resolve()
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from nstat import CIF, DecodingAlgorithms, History, Covariate, nspikeTrain, nstColl
from nstat.notebook_figures import FigureTracker

np.random.seed(0)
OUTPUT_ROOT = REPO_ROOT / "output" / "notebook_images"
__tracker = FigureTracker(topic="DecodingExampleWithHist", output_root=OUTPUT_ROOT, expected_count=2)

def _prepare_figure(matlab_line: str, *, figsize=(8.0, 4.5)):
    fig = __tracker.new_figure(matlab_line)
    fig.clear()
    fig.set_size_inches(*figsize)
    return fig

def _plot_raster(ax, spike_coll):
    for row in range(1, spike_coll.numSpikeTrains + 1):
        train = spike_coll.getNST(row)
        spikes = np.asarray(train.getSpikeTimes(), dtype=float).reshape(-1)
        if spikes.size:
            ax.vlines(spikes, row - 0.4, row + 0.4, color="k", linewidth=0.5)
    ax.set_ylabel("Neuron")
    ax.set_ylim(0.5, spike_coll.numSpikeTrains + 0.5)

def _plot_decoded_ci(ax, time, decoded, cov, stim, title):
    center = np.asarray(decoded, dtype=float).reshape(-1)
    spread = np.asarray(cov, dtype=float).reshape(-1)
    z_val = 3.0
    lower = center - z_val * spread
    upper = center + z_val * spread
    ax.plot(time[: center.size], center, "b", linewidth=1.5, label="x_{k|k}(t)")
    ax.plot(time[: center.size], lower, "g", linewidth=1.0, label="x_{k|k}(t)-3σ")
    ax.plot(time[: center.size], upper, "r", linewidth=1.0, label="x_{k|k}(t)+3σ")
    ax.plot(time[: center.size], np.asarray(stim).reshape(-1)[: center.size], "k", linewidth=1.5, label="x(t)")
    ax.set_title(title)
    ax.set_xlabel("time (s)")
    ax.legend(loc="upper right", frameon=False, fontsize=8)

def _simulate_history_spike_train(time, stim_data, baseline, hist_coeffs, window_times):
    spikes = []
    for idx in range(1, len(time)):
        t = time[idx]
        spike_arr = np.asarray(spikes, dtype=float)
        history_counts = []
        for w_start, w_stop in zip(window_times[:-1], window_times[1:]):
            if spike_arr.size:
                history_counts.append(np.sum((spike_arr >= t - w_stop) & (spike_arr < t - w_start)))
            else:
                history_counts.append(0.0)
        eta = baseline + stim_data[idx] + float(np.dot(hist_coeffs, history_counts))
        p = np.exp(np.clip(eta, -20.0, 20.0))
        p = p / (1.0 + p)
        if np.random.rand() < p:
            spikes.append(t)
    return np.asarray(spikes, dtype=float)

# SECTION 0: 1-D Stimulus Decode with History Effect
# We simulate neurons with refractory-history effects and compare point-process decoding with and without the correct history terms.

In [ ]:
# SECTION 1: History-aware decoding workflow
plt.close("all")
delta = 0.001
Tmax = 1.0
time = np.arange(0.0, Tmax + delta, delta)
f = 1.0
b1 = 1.0
b0 = -2.0
stimData = b1 * np.sin(2.0 * np.pi * f * time)
histCoeffs = np.array([-2.0, -2.0, -4.0])
windowTimes = np.array([0.0, 0.001, 0.002, 0.003])
histObj = History(windowTimes)
stim = Covariate(time, stimData, "Stimulus", "time", "s", "Voltage", ["sin"])

numRealizations = 20
trains = []
for idx in range(numRealizations):
    spikes = _simulate_history_spike_train(time, stimData, b0, histCoeffs, windowTimes)
    trains.append(nspikeTrain(spikes, str(idx + 1), delta, 0.0, Tmax, makePlots=-1))
sC = nstColl(trains)

fig = _prepare_figure("figure", figsize=(8.0, 5.5))
axs = fig.subplots(2, 1, sharex=True)
_plot_raster(axs[0], sC)
axs[0].set_title("History-dependent simulated spike trains")
axs[1].plot(time, stim.data[:, 0], color="k", linewidth=1.5)
axs[1].set_title("Stimulus")
axs[1].set_xlabel("time (s)")
axs[1].set_ylabel("Voltage")

lambdaCIF = [CIF([b0, b1], ["1", "x"], ["x"], "binomial", histCoeffs, histObj) for _ in range(numRealizations)]
lambdaCIFNoHist = [CIF([b0, b1], ["1", "x"], ["x"], "binomial") for _ in range(numRealizations)]

sC.resample(1.0 / delta)
dN = sC.dataToMatrix()
Q = 2.0 * np.std(np.diff(stim.data[:, 0]))
Px0 = 0.1
A = 1.0
x_p, W_p, x_u, W_u, *_ = DecodingAlgorithms.PPDecodeFilter(A, Q, Px0, dN.T, lambdaCIF, delta)
x_p_no_hist, W_p_no_hist, x_u_no_hist, W_u_no_hist, *_ = DecodingAlgorithms.PPDecodeFilter(
    A,
    Q,
    Px0,
    dN.T,
    lambdaCIFNoHist,
    delta,
)

fig = _prepare_figure("figure", figsize=(8.0, 6.0))
axs = fig.subplots(2, 1, sharex=True)
_plot_decoded_ci(axs[0], time, x_u, W_u, stim.data[:, 0], f"Decoded stimulus with history using {numRealizations} cells")
_plot_decoded_ci(axs[1], time, x_u_no_hist, W_u_no_hist, stim.data[:, 0], f"Decoded stimulus without history using {numRealizations} cells")
__tracker.finalize()